In [1]:
import pandas as pd

In [2]:
import os
print(os.getcwd())

/Users/tanya/PyCharmProjects/dear-neighbour/python


In [5]:
input_file = "../data/sosedushka.xlsx"
sheets = pd.read_excel(input_file, sheet_name=None)

#### Collect Russian column names and values

In [6]:
column_names = []

for sheet, data in sheets.items():
    print(sheet)
    column_names += data.columns.tolist()

README
fact_sales
fact_sales_dirty
dim_products
dim_stores
dim_customers
dim_suppliers


In [7]:
values = []

for sheet, data in sheets.items():
    if sheet != 'README' and sheet != 'fact_sales_dirty':
        for col_name in data.select_dtypes(include=['object', 'string']).columns:
            values += sheets[sheet][col_name].unique().tolist()

values = list(set(values))

In [8]:
print(len(values))

1075


In [ ]:
with open('russian_values.txt', 'w') as f:
    for word in values:
        f.write(word.lower() + '\n')

In [ ]:
with open('russian_columns.txt', 'w') as f:
    for word in column_names:
        f.write(word.lower().replace(' ', '_') + '\n')

#### Russian -> English dictionaries


In [ ]:
value_translation = {}

with open('russian_values.txt', 'r') as rf:
    with open('english_values.txt', 'r') as ef:
        for val, translation in zip(rf.read().splitlines(), ef.read().splitlines()):
            value_translation[val] = translation
            print()

len(value_translation)

In [21]:
value_translation['лук репчатый 1кг']

'onions 1kg'

In [22]:
column_translation = {}

with open('russian_columns.txt', 'r') as rf:
    with open('english_columns.txt', 'r') as ef:
        for val, translation in zip(rf.read().splitlines(), ef.read().splitlines()):
            column_translation[val] = translation

In [23]:
column_translation['фио']

'full_name'

#### Rebuilding the file with the columns and text values translated into English

In [ ]:
selected_sheets = [
    "fact_sales",
    "dim_products",
    "dim_stores",
    "dim_customers",
    "dim_suppliers"
]

translated_sheets = {}

for sheet_name in selected_sheets:
    df = sheets[sheet_name].copy()

    df.rename(
       columns=lambda col: column_translation.get(col.strip().replace(' ', '_').lower(), col),
        inplace=True
    )

    for column in df.columns:
        if df[column].dtype == "string":
            df[column] = df[column].map(
                lambda x: value_translation.get(x.lower(), x)
            )

    translated_sheets[sheet_name] = df

output_file = "../data/dear_neighbour.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in translated_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)